# 16 — Targeted Unlearning of Qwen Profile Memory

## Purpose

This notebook starts from the trained Qwen profile-memory model saved by Notebook 14.

Notebook 14 trained Qwen broadly on 14 profile fields, then reinforced six selected fields. This notebook tests and unlearns only those six reinforced fields:

- recipient age;
- recipient region;
- donor ID;
- donor age;
- kidney-failure cause; and
- dialysis duration.

## Experiment

1. Load the saved Notebook 14 model.
2. Split its 300 training recipients into:
   - 100 forget recipients; and
   - 200 retain recipients.
3. Test the original model manually and across all six fields.
4. Apply targeted gradient-difference unlearning to the six fields for the 100 forget recipients.
5. Test the same manual examples and full evaluation set again.
6. Compare before and after results.

This notebook does not retrain Qwen from scratch and does not make clinical-prediction claims.

## Before running

Notebook 14 must have completed successfully.

This notebook needs:

- the saved `qwen_profile_memory_model.tar.gz` archive from Notebook 14;
- `memory_training_recipient_ids.csv`;
- `profile_memory_contract.json`.

The archive may be uploaded to `/content`, or retrieved from the cloned repository through Git LFS.

In [1]:
%pip install -q -U unsloth

from pathlib import Path
from itertools import cycle
import json
import random
import tarfile
import time

import numpy as np
import pandas as pd
import torch

from unsloth import FastLanguageModel

SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

torch.cuda.manual_seed_all(SEED)

print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 129.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3

In [2]:
# Locate the repository and Notebook 14 artefacts.

REPO_OVERRIDE = None

repo_candidates = [
    Path("/content/qub-machine-unlearning"),
    Path.cwd(),
    Path.cwd().parent,
]

if REPO_OVERRIDE is not None:
    repo_candidates.insert(0, Path(REPO_OVERRIDE))

REPO_ROOT = next(
    (
        path
        for path in repo_candidates
        if (path / "code" / "final_submission").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Project repository not found. Clone it into /content first."
    )

FINAL_SUBMISSION_DIR = (
    REPO_ROOT / "code" / "final_submission"
)

DATA_DIR = FINAL_SUBMISSION_DIR / "data" / "final"
PROCESSED_DIR = FINAL_SUBMISSION_DIR / "processed_data"

ASSESSMENT_PATH = (
    DATA_DIR / "kidney_transplant_assessments.csv"
)

SPLIT_PATH = (
    PROCESSED_DIR / "split_assignments.csv"
)

NOTEBOOK14_RESULTS_DIR = (
    FINAL_SUBMISSION_DIR
    / "results"
    / "qwen_profile_memory"
)

UNLEARNING_RESULTS_DIR = (
    FINAL_SUBMISSION_DIR
    / "results"
    / "qwen_profile_memory_unlearning"
)

UNLEARNING_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CONTRACT_PATH = (
    NOTEBOOK14_RESULTS_DIR
    / "profile_memory_contract.json"
)

MEMORY_IDS_PATH = (
    NOTEBOOK14_RESULTS_DIR
    / "memory_training_recipient_ids.csv"
)

for required_path in [
    CONTRACT_PATH,
    MEMORY_IDS_PATH,
    ASSESSMENT_PATH,
    SPLIT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required file: {required_path}"
        )

with open(CONTRACT_PATH, encoding="utf-8") as handle:
    NOTEBOOK14_CONTRACT = json.load(handle)

FOCUSED_FIELDS = list(
    NOTEBOOK14_CONTRACT["focused_memory_fields"]
)

assert len(FOCUSED_FIELDS) == 6

QUESTION_SPECS = {
    field: NOTEBOOK14_CONTRACT["question_specs"][field]
    for field in FOCUSED_FIELDS
}

memory_recipient_ids = (
    pd.read_csv(MEMORY_IDS_PATH)["recipient_id"]
    .astype(str)
    .tolist()
)

assert len(memory_recipient_ids) == 300

# The fixed split is made once, before unlearning.
# First 100 people are forgotten; remaining 200 are retained.

forget_recipient_ids = memory_recipient_ids[:100]
retain_recipient_ids = memory_recipient_ids[100:]

assert len(forget_recipient_ids) == 100
assert len(retain_recipient_ids) == 200
assert not set(forget_recipient_ids).intersection(
    retain_recipient_ids
)

# Rebuild one deterministic profile per training recipient.

assessments = pd.read_csv(ASSESSMENT_PATH)
split_assignments = pd.read_csv(SPLIT_PATH)

data = assessments.merge(
    split_assignments[
        ["recipient_id", "donor_id", "split"]
    ],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)

train_df = data.loc[
    data["split"].eq("train")
].copy()

profiles = (
    train_df
    .sort_values(["recipient_id", "assessment_date"])
    .drop_duplicates("recipient_id", keep="first")
    .set_index("recipient_id")
)

profiles.index = profiles.index.astype(str)

# Locate and restore the saved Notebook 14 model.

MODEL_ARCHIVE_CANDIDATES = [
    Path("/content/qwen_profile_memory_model.tar.gz"),
    (
        FINAL_SUBMISSION_DIR
        / "models"
        / "qwen_profile_memory"
        / "qwen_profile_memory_model.tar.gz"
    ),
    (
        FINAL_SUBMISSION_DIR
        / "models"
        / "qwen"
        / "qwen_profile_memory_model.tar.gz"
    ),
]

ORIGINAL_ARCHIVE = next(
    (
        path
        for path in MODEL_ARCHIVE_CANDIDATES
        if path.exists() and path.stat().st_size > 1_000_000
    ),
    None,
)

if ORIGINAL_ARCHIVE is None:
    raise FileNotFoundError(
        "Notebook 14 model archive not found. Upload "
        "qwen_profile_memory_model.tar.gz to /content, "
        "or run git lfs pull in the repository."
    )

ORIGINAL_MODEL_DIR = Path(
    "/content/qwen_profile_memory_model"
)

if not ORIGINAL_MODEL_DIR.exists():
    with tarfile.open(ORIGINAL_ARCHIVE, "r:gz") as archive:
        archive.extractall("/content")

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ORIGINAL_MODEL_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,
)

tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False

print("Repository:", REPO_ROOT)
print("Original model:", ORIGINAL_MODEL_DIR)
print("Forget recipients:", len(forget_recipient_ids))
print("Retain recipients:", len(retain_recipient_ids))
print("Six focused fields:", FOCUSED_FIELDS)

/tmp/ipykernel_3993/109718021.py:182: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall("/content")


==((====))==  Unsloth 2026.9.3: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.9.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Repository: /content/qub-machine-unlearning
Original model: /content/qwen_profile_memory_model
Forget recipients: 100
Retain recipients: 200
Six focused fields: ['recipient_age', 'recipient_region', 'donor_id', 'donor_age', 'kidney_failure_cause', 'dialysis_months']


## Create the fixed unlearning and evaluation data

The same six fields are used everywhere in this notebook:

- baseline testing;
- forget objective;
- retain safety objective;
- post-unlearning testing.

This keeps the experiment consistent.

In [3]:
INTEGER_FIELDS = {
    "recipient_age",
    "donor_age",
    "dialysis_months",
}

def format_answer(value, field):
    if field in INTEGER_FIELDS:
        return str(int(value))
    return str(value)

def build_prompt(recipient_id, field):
    return f"""Here is the recipient ID:
{recipient_id}

{QUESTION_SPECS[field]}

SOLUTION
"""

def normalise_answer(value):
    return " ".join(
        str(value).strip().casefold().split()
    )

def make_examples(recipient_ids, fields):
    rows = []

    for recipient_id in recipient_ids:
        for field in fields:
            recorded_answer = format_answer(
                profiles.loc[recipient_id, field],
                field,
            )

            rows.append({
                "recipient_id": recipient_id,
                "field": field,
                "recorded_answer": recorded_answer,
                "text": (
                    build_prompt(recipient_id, field)
                    + recorded_answer
                    + tokenizer.eos_token
                ),
            })

    return pd.DataFrame(rows)

# Data used during gradient-difference unlearning.

forget_examples = make_examples(
    forget_recipient_ids,
    FOCUSED_FIELDS,
)

retain_examples = make_examples(
    retain_recipient_ids,
    FOCUSED_FIELDS,
)

assert len(forget_examples) == 600
assert len(retain_examples) == 1_200

# Fixed evaluation plan used before and after unlearning.

evaluation_rows = []

for group, recipient_ids in [
    ("forget", forget_recipient_ids),
    ("retain", retain_recipient_ids),
]:
    for recipient_id in recipient_ids:
        for field in FOCUSED_FIELDS:
            evaluation_rows.append({
                "group": group,
                "recipient_id": recipient_id,
                "field": field,
                "recorded_answer": format_answer(
                    profiles.loc[recipient_id, field],
                    field,
                ),
            })

EVALUATION_PLAN = pd.DataFrame(evaluation_rows)

assert len(EVALUATION_PLAN) == 1_800

# Fixed manual examples. These never change after unlearning.

MANUAL_FIELD = FOCUSED_FIELDS[0]

MANUAL_PLAN = pd.DataFrame([
    {
        "group": "forget",
        "recipient_id": forget_recipient_ids[0],
        "field": MANUAL_FIELD,
    },
    {
        "group": "retain",
        "recipient_id": retain_recipient_ids[0],
        "field": MANUAL_FIELD,
    },
])

MANUAL_PLAN["recorded_answer"] = MANUAL_PLAN.apply(
    lambda row: format_answer(
        profiles.loc[row["recipient_id"], row["field"]],
        row["field"],
    ),
    axis=1,
)

display(MANUAL_PLAN)

print("Forget questions:", len(forget_examples))
print("Retain questions:", len(retain_examples))
print("Evaluation questions:", len(EVALUATION_PLAN))

,group,recipient_id,field,recorded_answer
0,forget,V32P-R001887,recipient_age,48
1,retain,V32P-R002714,recipient_age,57


Forget questions: 600
Retain questions: 1200
Evaluation questions: 1800


In [4]:
def generate_answer(model, tokenizer, prompt):
    FastLanguageModel.for_inference(model)

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **encoded,
            max_new_tokens=12,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[
        0,
        encoded["input_ids"].shape[1]:,
    ]

    return (
        tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )
        .strip()
        .splitlines()[0]
        .strip()
    )

def correct_answer_log_probability(
    model,
    tokenizer,
    prompt,
    recorded_answer,
):
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"].to(model.device)

    full_ids = tokenizer(
        prompt + recorded_answer,
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"].to(model.device)

    answer_start = prompt_ids.shape[1]

    with torch.inference_mode():
        logits = model(input_ids=full_ids).logits

    token_log_probabilities = []

    for offset, token_id in enumerate(
        full_ids[0, answer_start:]
    ):
        token_log_probabilities.append(
            torch.log_softmax(
                logits[0, answer_start + offset - 1],
                dim=-1,
            )[token_id].item()
        )

    return float(np.mean(token_log_probabilities))

def evaluate(model, tokenizer, plan, output_path):
    key_columns = ["group", "recipient_id", "field"]

    if output_path.exists():
        existing_rows = pd.read_csv(output_path)

        if set(key_columns).issubset(existing_rows.columns):
            existing_rows = (
                existing_rows
                .drop_duplicates(key_columns, keep="last")
                .copy()
            )
        else:
            existing_rows = pd.DataFrame()
    else:
        existing_rows = pd.DataFrame()

    completed_keys = set()

    if not existing_rows.empty:
        completed_keys = set(
            existing_rows[key_columns].itertuples(
                index=False,
                name=None,
            )
        )

    new_rows = []

    for _, row in plan.iterrows():
        result_key = (
            row["group"],
            row["recipient_id"],
            row["field"],
        )

        if result_key in completed_keys:
            continue

        prompt = build_prompt(
            row["recipient_id"],
            row["field"],
        )

        generated_answer = generate_answer(
            model,
            tokenizer,
            prompt,
        )

        result = row.to_dict()

        result.update({
            "generated_answer": generated_answer,
            "exact_match": (
                normalise_answer(generated_answer)
                == normalise_answer(
                    row["recorded_answer"]
                )
            ),
            "correct_answer_log_probability": (
                correct_answer_log_probability(
                    model,
                    tokenizer,
                    prompt,
                    row["recorded_answer"],
                )
            ),
        })

        new_rows.append(result)

        if len(new_rows) % 25 == 0:
            saved_rows = pd.concat(
                [
                    existing_rows,
                    pd.DataFrame(new_rows),
                ],
                ignore_index=True,
            )

            saved_rows.to_csv(
                output_path,
                index=False,
            )

            print(
                "Saved",
                len(saved_rows),
                "of",
                len(plan),
                "questions",
            )

    results = pd.concat(
        [
            existing_rows,
            pd.DataFrame(new_rows),
        ],
        ignore_index=True,
    )

    results = (
        results
        .drop_duplicates(key_columns, keep="last")
        .sort_values(key_columns)
        .reset_index(drop=True)
    )

    results.to_csv(output_path, index=False)

    assert len(results) == len(plan)

    return results

def summarise_results(rows):
    summary = (
        rows
        .groupby("group", as_index=False)
        .agg(
            recipients=(
                "recipient_id",
                "nunique",
            ),
            questions=(
                "exact_match",
                "size",
            ),
            correct_answers=(
                "exact_match",
                "sum",
            ),
            exact_recall=(
                "exact_match",
                "mean",
            ),
            mean_correct_answer_log_probability=(
                "correct_answer_log_probability",
                "mean",
            ),
        )
    )

    summary["exact_recall_pct"] = (
        100 * summary["exact_recall"]
    ).round(2)

    return summary

## Baseline manual examples

The original Notebook 14 model is asked one forget question and one retain question before unlearning.

The same recipient IDs and questions will be used again after unlearning.

In [5]:
def run_manual_examples(model, tokenizer):
    rows = []

    for _, example in MANUAL_PLAN.iterrows():
        prompt = build_prompt(
            example["recipient_id"],
            example["field"],
        )

        generated_answer = generate_answer(
            model,
            tokenizer,
            prompt,
        )

        rows.append({
            "group": example["group"],
            "recipient_id": example["recipient_id"],
            "field": example["field"],
            "recorded_answer": example["recorded_answer"],
            "model_answer": generated_answer,
            "exact_match": (
                normalise_answer(generated_answer)
                == normalise_answer(
                    example["recorded_answer"]
                )
            ),
        })

    return pd.DataFrame(rows)

manual_before = run_manual_examples(
    model,
    tokenizer,
)

print("Original Notebook 14 model: manual examples")
display(manual_before)

Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original Notebook 14 model: manual examples


,group,recipient_id,field,recorded_answer,model_answer,exact_match
0,forget,V32P-R001887,recipient_age,48,48,True
1,retain,V32P-R002714,recipient_age,57,57,True


## Baseline full results

The unchanged Notebook 14 model is tested on:

- 100 forget recipients × six fields = 600 questions;
- 200 retain recipients × six fields = 1,200 questions.

These are the before-unlearning results.

In [6]:
RUN_NAME = "targeted_six_field_unlearning_v1"

ORIGINAL_RESULTS_PATH = (
    UNLEARNING_RESULTS_DIR
    / f"original_forget_retain_{RUN_NAME}.csv"
)

original_rows = evaluate(
    model,
    tokenizer,
    EVALUATION_PLAN,
    ORIGINAL_RESULTS_PATH,
)

original_summary = summarise_results(
    original_rows
)

print("Original-model baseline results")
display(original_summary)

print("Saved:", ORIGINAL_RESULTS_PATH)

Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
`use_return_dict` is deprecated! Use `return_dict` instead!
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Saved 25 of 1800 questions


Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Saved 50 of 1800 questions


Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

...

Saved 1775 of 1800 questions


Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Saved 1800 of 1800 questions
Original-model baseline results


,group,recipients,questions,correct_answers,exact_recall,mean_correct_answer_log_probability,exact_recall_pct
0,forget,100,600,587,0.978333,-0.034375,97.83
1,retain,200,1200,1183,0.985833,-0.028055,98.58


Saved: /content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory_unlearning/original_forget_retain_targeted_six_field_unlearning_v1.csv


## Targeted gradient-difference unlearning

The model is updated for 200 steps.

- The forget loss is increased for six fields belonging to the 100 forget recipients.
- The retain loss is minimised for the same six fields belonging to the 200 retain recipients.
- The safest checkpoint is selected using retain loss only.

No evaluation result is used to select the checkpoint.

In [7]:
def labels_after_solution(
    tokenizer,
    input_ids,
    attention_mask,
):
    labels = input_ids.clone()

    solution_marker = tokenizer.encode(
        "SOLUTION\n",
        add_special_tokens=False,
    )

    for row_index in range(input_ids.shape[0]):
        token_ids = input_ids[row_index].tolist()

        marker_start = next(
            (
                index
                for index in range(
                    len(token_ids)
                    - len(solution_marker)
                    + 1
                )
                if token_ids[
                    index:index + len(solution_marker)
                ] == solution_marker
            ),
            None,
        )

        if marker_start is None:
            raise RuntimeError(
                "SOLUTION marker not found."
            )

        labels[
            row_index,
            :marker_start + len(solution_marker),
        ] = -100

    labels[attention_mask == 0] = -100

    return labels

def batch_loss(model, tokenizer, texts):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    ).to(model.device)

    labels = labels_after_solution(
        tokenizer,
        encoded["input_ids"],
        encoded["attention_mask"],
    )

    return model(
        **encoded,
        labels=labels,
    ).loss

def targeted_unlearn(
    model,
    tokenizer,
    forget_texts,
    retain_texts,
    steps=200,
    learning_rate=2e-5,
    retain_weight=1.0,
):
    FastLanguageModel.for_training(model)
    model.train()

    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    optimiser = torch.optim.AdamW(
        trainable_parameters,
        lr=learning_rate,
    )

    baseline_retain_loss = float(
        batch_loss(
            model,
            tokenizer,
            retain_texts[:4],
        )
        .detach()
        .cpu()
    )

    forget_cycle = cycle(forget_texts)
    retain_cycle = cycle(retain_texts)

    best_state = None
    best_forget_loss = -float("inf")
    history_rows = []

    for step in range(1, steps + 1):
        forget_batch = [
            next(forget_cycle)
            for _ in range(4)
        ]

        retain_batch = [
            next(retain_cycle)
            for _ in range(4)
        ]

        forget_loss = batch_loss(
            model,
            tokenizer,
            forget_batch,
        )

        retain_loss = batch_loss(
            model,
            tokenizer,
            retain_batch,
        )

        objective = (
            -forget_loss
            + retain_weight * retain_loss
        )

        optimiser.zero_grad(set_to_none=True)
        objective.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            1.0,
        )

        optimiser.step()

        row = {
            "step": step,
            "forget_loss": float(
                forget_loss.detach().cpu()
            ),
            "retain_loss": float(
                retain_loss.detach().cpu()
            ),
        }

        history_rows.append(row)

        if (
            row["retain_loss"]
            <= baseline_retain_loss * 1.10
            and row["forget_loss"] > best_forget_loss
        ):
            best_forget_loss = row["forget_loss"]

            best_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter
                in model.named_parameters()
                if parameter.requires_grad
            }

    if best_state is None:
        raise RuntimeError(
            "No checkpoint met the retain-loss safety rule."
        )

    for name, parameter in model.named_parameters():
        if name in best_state:
            parameter.data.copy_(
                best_state[name].to(parameter.device)
            )

    return (
        pd.DataFrame(history_rows),
        baseline_retain_loss,
    )

started = time.perf_counter()

unlearning_history, baseline_retain_loss = (
    targeted_unlearn(
        model,
        tokenizer,
        forget_examples["text"].tolist(),
        retain_examples["text"].tolist(),
        steps=200,
        learning_rate=2e-5,
        retain_weight=1.0,
    )
)

unlearning_seconds = (
    time.perf_counter() - started
)

UNLEARNED_MODEL_DIR = (
    Path("/content")
    / "targeted_unlearned_qwen_profile_memory"
)

UNLEARNED_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model.save_pretrained(UNLEARNED_MODEL_DIR)
tokenizer.save_pretrained(UNLEARNED_MODEL_DIR)

UNLEARNED_ARCHIVE = (
    UNLEARNED_MODEL_DIR.with_suffix(".tar.gz")
)

with tarfile.open(
    UNLEARNED_ARCHIVE,
    "w:gz",
) as archive:
    archive.add(
        UNLEARNED_MODEL_DIR,
        arcname=UNLEARNED_MODEL_DIR.name,
    )

HISTORY_PATH = (
    UNLEARNING_RESULTS_DIR
    / f"unlearning_history_{RUN_NAME}.csv"
)

RUNTIME_PATH = (
    UNLEARNING_RESULTS_DIR
    / f"unlearning_runtime_{RUN_NAME}.csv"
)

unlearning_history.to_csv(
    HISTORY_PATH,
    index=False,
)

pd.DataFrame([{
    "steps": 200,
    "learning_rate": 2e-5,
    "retain_weight": 1.0,
    "baseline_retain_loss": baseline_retain_loss,
    "seconds": unlearning_seconds,
}]).to_csv(
    RUNTIME_PATH,
    index=False,
)

print("Saved unlearned model:", UNLEARNED_MODEL_DIR)
print("Saved archive:", UNLEARNED_ARCHIVE)
print("Saved history:", HISTORY_PATH)

Unsloth: Restored added_tokens_decoder metadata in /content/targeted_unlearned_qwen_profile_memory/tokenizer_config.json.


Saved unlearned model: /content/targeted_unlearned_qwen_profile_memory
Saved archive: /content/targeted_unlearned_qwen_profile_memory.tar.gz
Saved history: /content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory_unlearning/unlearning_history_targeted_six_field_unlearning_v1.csv


## Post-unlearning manual checks

Test the same forget and retain examples used before unlearning. The forget answer should become less accurate, while the retain answer should remain accurate.

In [9]:
# Run the same manual examples using the targeted-unlearned model.

manual_after = run_manual_examples(
    model,
    tokenizer,
)

# Place the original and unlearned answers side by side.

manual_comparison = manual_before.merge(
    manual_after,
    on=[
        "group",
        "recipient_id",
        "field",
        "recorded_answer",
    ],
    suffixes=("_before", "_after"),
    validate="one_to_one",
)

display(manual_comparison)

Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,group,recipient_id,field,recorded_answer,model_answer_before,exact_match_before,model_answer_after,exact_match_after
0,forget,V32P-R001887,recipient_age,48,48,True,63,False
1,retain,V32P-R002714,recipient_age,57,57,True,57,True


## Post-unlearning quantitative evaluation

Evaluate the unlearned model on exactly the same 100 forget recipients and 200 retain recipients used for the original-model baseline.

In [10]:
def summarise_model_results(rows):
    """Create one clear results row for each recipient group."""

    summary = (
        rows
        .groupby("group", as_index=False)
        .agg(
            recipients=("recipient_id", "nunique"),
            questions=("exact_match", "size"),
            correct_answers=("exact_match", "sum"),
            exact_recall=("exact_match", "mean"),
            mean_correct_answer_log_probability=(
                "correct_answer_log_probability",
                "mean",
            ),
        )
    )

    summary["exact_recall_pct"] = (
        100 * summary["exact_recall"]
    ).round(2)

    return summary


# Evaluate the unlearned model using the fixed before/after plan.

UNLEARNED_RESULTS_PATH = (
    UNLEARNING_RESULTS_DIR
    / f"unlearned_forget_retain_{RUN_NAME}.csv"
)

unlearned_rows = evaluate(
    model,
    tokenizer,
    EVALUATION_PLAN,
    UNLEARNED_RESULTS_PATH,
)

unlearned_summary = summarise_model_results(unlearned_rows)

print("Targeted-unlearned model results")
display(unlearned_summary)

print("Saved:", UNLEARNED_RESULTS_PATH)

Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Saved 25 of 1800 questions


Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Saved 50 of 1800 questions


Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

...

Saved 1775 of 1800 questions


Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Saved 1800 of 1800 questions
Targeted-unlearned model results


,group,recipients,questions,correct_answers,exact_recall,mean_correct_answer_log_probability,exact_recall_pct
0,forget,100,600,376,0.626667,-1.598578,62.67
1,retain,200,1200,1117,0.930833,-0.091100,93.08


Saved: /content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory_unlearning/unlearned_forget_retain_targeted_six_field_unlearning_v1.csv


## Before-versus-after targeted unlearning comparison

A successful result means forget recall and answer confidence decrease, while retain recall remains high.

In [11]:
# Recreate the original-model summary to ensure both results use
# exactly the same calculation.

original_summary = summarise_model_results(original_rows)

# Compare the original and targeted-unlearned model.

before_after_comparison = original_summary.merge(
    unlearned_summary,
    on="group",
    suffixes=("_before", "_after"),
    validate="one_to_one",
)

# Negative recall change is desired for the forget group.
# A value near zero is desired for the retain group.

before_after_comparison["recall_change_pp"] = (
    100
    * (
        before_after_comparison["exact_recall_after"]
        - before_after_comparison["exact_recall_before"]
    )
).round(2)

# Negative values mean the model assigns less probability
# to the previously correct profile answer.

before_after_comparison["answer_log_probability_change"] = (
    before_after_comparison[
        "mean_correct_answer_log_probability_after"
    ]
    - before_after_comparison[
        "mean_correct_answer_log_probability_before"
    ]
).round(4)

before_after_comparison["desired_result"] = (
    before_after_comparison["group"].map(
        {
            "forget": "Recall should decrease",
            "retain": "Recall should remain high",
        }
    )
)

before_after_comparison = (
    before_after_comparison
    .sort_values("group")
    .reset_index(drop=True)
)

COMPARISON_PATH = (
    UNLEARNING_RESULTS_DIR
    / f"before_after_targeted_unlearning_{RUN_NAME}.csv"
)

before_after_comparison.to_csv(
    COMPARISON_PATH,
    index=False,
)

display(before_after_comparison)

print("Saved comparison:", COMPARISON_PATH)

,group,recipients_before,questions_before,correct_answers_before,exact_recall_before,mean_correct_answer_log_probability_before,exact_recall_pct_before,recipients_after,questions_after,correct_answers_after,exact_recall_after,mean_correct_answer_log_probability_after,exact_recall_pct_after,recall_change_pp,answer_log_probability_change,desired_result
0,forget,100,600,587,0.978333,-0.034375,97.83,100,600,376,0.626667,-1.598578,62.67,-35.17,-1.5642,Recall should decrease
1,retain,200,1200,1183,0.985833,-0.028055,98.58,200,1200,1117,0.930833,-0.091100,93.08,-5.50,-0.0630,Recall should remain high


Saved comparison: /content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory_unlearning/before_after_targeted_unlearning_targeted_six_field_unlearning_v1.csv


## Final before-versus-after comparison

A successful targeted-unlearning result should show:

- lower forget-group recall after unlearning; and
- retain-group recall that stays as close as possible to its original level.

In [12]:
comparison = original_summary.merge(
    unlearned_summary,
    on=["group", "recipients", "questions"],
    suffixes=("_before", "_after"),
    validate="one_to_one",
)

comparison["recall_change_percentage_points"] = (
    100
    * (
        comparison["exact_recall_after"]
        - comparison["exact_recall_before"]
    )
).round(2)

comparison["log_probability_change"] = (
    comparison[
        "mean_correct_answer_log_probability_after"
    ]
    - comparison[
        "mean_correct_answer_log_probability_before"
    ]
).round(4)

comparison["desired_result"] = comparison["group"].map({
    "forget": "Recall should decrease",
    "retain": "Recall should stay high",
})

comparison = (
    comparison
    .sort_values("group")
    .reset_index(drop=True)
)

COMPARISON_PATH = (
    UNLEARNING_RESULTS_DIR
    / f"before_after_summary_{RUN_NAME}.csv"
)

comparison.to_csv(
    COMPARISON_PATH,
    index=False,
)

display(comparison)

checks = {
    "Original model loaded": ORIGINAL_MODEL_DIR.exists(),
    "100 forget recipients": len(forget_recipient_ids) == 100,
    "200 retain recipients": len(retain_recipient_ids) == 200,
    "Six fields used throughout": len(FOCUSED_FIELDS) == 6,
    "Original results saved": ORIGINAL_RESULTS_PATH.exists(),
    "Unlearned model saved": UNLEARNED_MODEL_DIR.exists(),
    "Unlearned archive saved": UNLEARNED_ARCHIVE.exists(),
    "Unlearned results saved": UNLEARNED_RESULTS_PATH.exists(),
    "Comparison saved": COMPARISON_PATH.exists(),
}

display(pd.Series(checks).to_frame("Pass"))

assert all(checks.values())

print("Saved final comparison:", COMPARISON_PATH)

,group,recipients,questions,correct_answers_before,exact_recall_before,mean_correct_answer_log_probability_before,exact_recall_pct_before,correct_answers_after,exact_recall_after,mean_correct_answer_log_probability_after,exact_recall_pct_after,recall_change_percentage_points,log_probability_change,desired_result
0,forget,100,600,587,0.978333,-0.034375,97.83,376,0.626667,-1.598578,62.67,-35.17,-1.5642,Recall should decrease
1,retain,200,1200,1183,0.985833,-0.028055,98.58,1117,0.930833,-0.091100,93.08,-5.50,-0.0630,Recall should stay high


,Pass
Original model loaded,True
100 forget recipients,True
200 retain recipients,True
Six fields used throughout,True
Original results saved,True
Unlearned model saved,True
Unlearned archive saved,True
Unlearned results saved,True
Comparison saved,True


Saved final comparison: /content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory_unlearning/before_after_summary_targeted_six_field_unlearning_v1.csv
